# FICTURE F1 metric

Agreement between **FICTURE factor identity** and a segmentation's **cell types**.
For every transcript we assign two canonical (parent-type) labels and score them as a
hard-label classification:

- **pred** = cell type of the nearest FICTURE pixel's top factor `K1` (within 5 um)
- **true** = cell type of the segmentation cell whose polygon the transcript lies in

`compute_ficture_f1` writes `metrics/<cohort>/ficture/ficture_f1.csv` with columns
`method, sample, cell_type, tp, fp, fn, precision, recall, f1`. Each sample block has
one row per cell type plus four summary rows (`f1_{micro,macro}_{all,vascular}`);
`sample == "all_samples"` holds the counts pooled across samples.

**How to plot this.** The prediction is a hard argmax factor label with no retained
confidence, so there is no score to threshold: **a precision-recall *curve* does not
apply** (it would collapse to one point per class). The right views are therefore the
single operating point per method/class plus per-type resolution:
1. macro & micro F1 per method (headline ranking),
2. per-cell-type F1 heatmap (where methods differ),
3. precision-vs-recall scatter of each method's operating point against F1 iso-contours
   (the correct substitute for a PR curve),
4. per-sample spread of macro F1 (stability).

## Compute FICTURE F1

In [ ]:
from cellseg_benchmark.metrics import (
    compute_ficture_f1,
    compute_metric_for_all_methods,
)

compute_metric_for_all_methods(
    compute_ficture_f1,
    results_name="ficture/ficture_f1.csv",
    cohort="aging",
    celltype_col="cell_type_revised",
    overwrite=False,
)

## Load results

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from cellseg_benchmark import BASE_PATH
from cellseg_benchmark._constants import method_colors

cohort = "aging"
results_file = Path(BASE_PATH) / "metrics" / cohort / "ficture" / "ficture_f1.csv"
plot_path = results_file.parent / "plots"
plot_path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(results_file, index_col=0)

summary_rows = ["f1_micro_all", "f1_macro_all", "f1_micro_vascular", "f1_macro_vascular"]
pooled = df[df["sample"] == "all_samples"]                      # pooled over samples
per_type = pooled[~pooled["cell_type"].isin(summary_rows)]      # one row per cell type
summ = pooled[pooled["cell_type"].isin(summary_rows)]           # the four F1 summaries

# rank methods by pooled macro F1 (best first); reuse as consistent order everywhere
order = (
    summ[summ["cell_type"] == "f1_macro_all"]
    .set_index("method")["f1"].sort_values(ascending=False).index.tolist()
)
pal = {m: method_colors.get(m, "0.6") for m in order}
df.head()

## 1. Method ranking: macro & micro F1

In [ ]:
# macro = unweighted mean over cell types; micro = abundance-weighted (pooled counts).
# shown for all cell types and for the vascular subset (ECs/Pericytes/SMCs/VLMCs).
plot_df = summ.copy()
plot_df["flavor"] = plot_df["cell_type"].str.replace("f1_", "", regex=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, scope in zip(axes, ["all", "vascular"]):
    sub = plot_df[plot_df["flavor"].str.endswith(scope)]
    sns.barplot(
        data=sub, x="method", y="f1",
        hue="flavor", order=order, ax=ax,
        palette={f"micro_{scope}": "#4C72B0", f"macro_{scope}": "#DD8452"},
    )
    ax.set_title(f"FICTURE F1 ({scope})")
    ax.set_ylim(0, 1)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
axes[0].set_ylabel("F1")
plt.tight_layout()
plt.savefig(plot_path / "ficture_f1_method_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Per-cell-type F1 (methods x cell types)

In [ ]:
mat = per_type.pivot(index="method", columns="cell_type", values="f1").loc[order]
mat = mat.reindex(columns=mat.mean().sort_values(ascending=False).index)  # easy types left

fig, ax = plt.subplots(figsize=(max(10, 0.6 * mat.shape[1]), 0.5 * mat.shape[0] + 2))
sns.heatmap(mat, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1,
            linewidths=0.5, cbar_kws={"label": "F1"}, ax=ax)
ax.set_title("FICTURE F1 per cell type (pooled over samples)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(plot_path / "ficture_f1_celltype_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Precision vs recall (operating point per method)

One point per method (pooled micro counts) rather than a swept PR curve, since labels
are hard. Grey lines are F1 iso-contours: points on the same curve share an F1, so
distance from the top-right corner is a direct read of segmentation quality.

In [ ]:
pr = summ[summ["cell_type"] == "f1_micro_all"]

fig, ax = plt.subplots(figsize=(6.5, 6.5))
# F1 iso-contours: precision = f*r / (2r - f)
r = np.linspace(0.01, 1, 200)
for f in [0.2, 0.4, 0.6, 0.8]:
    p = np.where(2 * r - f > 0, f * r / (2 * r - f), np.nan)
    p[(p < 0) | (p > 1)] = np.nan
    ax.plot(r, p, color="0.8", lw=1, zorder=0)
    ax.annotate(f"F1={f}", (r[~np.isnan(p)][-1], p[~np.isnan(p)][-1]),
                color="0.6", fontsize=8)

for _, row in pr.iterrows():
    ax.scatter(row["recall"], row["precision"], s=80,
               color=method_colors.get(row["method"], "0.4"),
               edgecolor="k", linewidth=0.5, label=row["method"], zorder=3)

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Micro precision vs recall (pooled)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(plot_path / "ficture_f1_precision_recall.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Per-sample spread of macro F1 (stability)

In [ ]:
per_sample = df[(df["sample"] != "all_samples") & (df["cell_type"] == "f1_macro_all")]

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=per_sample, x="method", y="f1", order=order,
            hue="method", palette=pal, legend=False, fliersize=0, ax=ax)
sns.stripplot(data=per_sample, x="method", y="f1", order=order,
              color="0.2", size=4, jitter=0.2, ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("macro F1 (per sample)")
ax.set_xlabel("")
ax.set_title("FICTURE macro F1 across samples")
ax.tick_params(axis="x", rotation=45)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
plt.tight_layout()
plt.savefig(plot_path / "ficture_f1_per_sample.png", dpi=150, bbox_inches="tight")
plt.show()